<a href="https://colab.research.google.com/github/ilfpns/HailMary_AI/blob/main/HM_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import importlib.util
import subprocess
import sys
import os
import re
import shutil

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

import torch
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path
from PIL import Image, ImageDraw
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from PIL import Image, ImageOps

import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

MODULE_DIR = Path("/content/drive/MyDrive/HM_img2")
MODULE = MODULE_DIR / "hm_data.py"

SOURCE = '''import shutil, subprocess
from pathlib import Path

ZIP = Path("/content/drive/MyDrive/HM_img2/HM_img2.zip")
EXTRACT = Path("/content/data")


def prepare_data(zip_path=ZIP, extract=EXTRACT, verbose=True):
    zip_path, extract = Path(zip_path), Path(extract)

    if not any(extract.rglob("data.yaml")):
        assert zip_path.exists(), f"zip이 없습니다: {zip_path}"
        if verbose:
            print("압축 해제:", zip_path.name, f"{zip_path.stat().st_size / 1024**3:.2f} GB")
        local_zip = Path("/content/dataset.zip")
        shutil.copy(zip_path, local_zip)
        extract.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(local_zip), "-d", str(extract)], check=True)
        local_zip.unlink()
    elif verbose:
        print("이미 풀려 있음:", extract)

    root = next(extract.rglob("data.yaml")).parent
    yaml_path = root / "data.yaml"
    lines = [l for l in yaml_path.read_text().splitlines() if not l.startswith("path:")]
    yaml_path.write_text("\\n".join([f"path: {root}"] + lines) + "\\n")

    if verbose:
        print("root:", root)
        for s in ["train", "valid", "test"]:
            if (root / s).exists():
                n_img = len(list((root / s / "images").iterdir()))
                n_lbl = len(list((root / s / "labels").iterdir()))
                print(f"{s:<6} 이미지 {n_img}  라벨 {n_lbl}")

    return root, yaml_path
'''

if MODULE.exists():
    print("모듈 있음:", MODULE)
else:
    MODULE_DIR.mkdir(parents=True, exist_ok=True)
    MODULE.write_text(SOURCE, encoding="utf-8")
    print("모듈 생성:", MODULE)

if str(MODULE_DIR) not in sys.path:
    sys.path.append(str(MODULE_DIR))
from hm_data import prepare_data

root, yaml_path = prepare_data()

Mounted at /content/drive
모듈 있음: /content/drive/MyDrive/HM_img2/hm_data.py
압축 해제: HM_img2.zip 22.24 GB
root: /content/data/HM_img2
train  이미지 20885  라벨 20885
valid  이미지 1531  라벨 1531
test   이미지 356  라벨 356


In [6]:
MAX = 640
root = next(Path("/content/data").rglob("data.yaml")).parent
paths = [p for s in ["train", "valid", "test"] for p in (root / s / "images").iterdir()]


def shrink(p):
    with Image.open(p) as im:
        size = im.size
        if max(size) <= MAX:
            return None
        im.draft("RGB", (MAX, MAX))
        out = ImageOps.exif_transpose(im).convert("RGB")
    out.thumbnail((MAX, MAX), Image.LANCZOS)
    out.save(p, quality=95)
    return f"{p.parent.parent.name}\t{p.stem}\t{size[0]}\t{size[1]}"


with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    large = [r for r in ex.map(shrink, paths, chunksize=64) if r]
print(f"줄인 이미지: {len(large)} / {len(paths)}")

Path("/content/drive/MyDrive/HM_img2/large_images.tsv").write_text("split\tstem\tw\th\n" + "\n".join(large) + "\n")

for c in root.rglob("*.cache"):
    c.unlink()

subprocess.run("cd /content/data && zip -qr -0 /content/HM_img2_640.zip .", shell=True, check=True)
shutil.copy("/content/HM_img2_640.zip", "/content/drive/MyDrive/HM_img2/HM_img2_640.zip")
print("백업 완료:", round(os.path.getsize("/content/HM_img2_640.zip") / 1024**3, 2), "GB")

KeyboardInterrupt: 

In [7]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4"], check=True)

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 → 런타임 유형 변경 → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/HM_img2")
ZIP = BASE / "HM_img2_640.zip"
MODULE = BASE / "hm_data.py"
PROJECT = BASE / "runs"
NAME = "HM_yolo26n"
MODEL = "yolo26n.pt"
EPOCHS = 100

SOURCE = '''import shutil, subprocess
from pathlib import Path

ZIP = Path("/content/drive/MyDrive/HM_img2/HM_img2_640.zip")
EXTRACT = Path("/content/data")


def prepare_data(zip_path=ZIP, extract=EXTRACT, verbose=True):
    zip_path, extract = Path(zip_path), Path(extract)

    if not any(extract.rglob("data.yaml")):
        assert zip_path.exists(), f"zip이 없습니다: {zip_path}"
        if verbose:
            print("압축 해제:", zip_path.name, f"{zip_path.stat().st_size / 1024**3:.2f} GB")
        local_zip = Path("/content/dataset.zip")
        shutil.copy(zip_path, local_zip)
        extract.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(local_zip), "-d", str(extract)], check=True)
        local_zip.unlink()
    elif verbose:
        print("이미 풀려 있음:", extract)

    root = next(extract.rglob("data.yaml")).parent
    yaml_path = root / "data.yaml"
    lines = [l for l in yaml_path.read_text().splitlines() if not l.startswith("path:")]
    yaml_path.write_text("\\n".join([f"path: {root}"] + lines) + "\\n")

    if verbose:
        print("root:", root)
        for s in ["train", "valid", "test"]:
            if (root / s).exists():
                n_img = len(list((root / s / "images").iterdir()))
                n_lbl = len(list((root / s / "labels").iterdir()))
                print(f"{s:<6} 이미지 {n_img}  라벨 {n_lbl}")

    return root, yaml_path
'''

if not MODULE.exists():
    BASE.mkdir(parents=True, exist_ok=True)
    MODULE.write_text(SOURCE, encoding="utf-8")
    print("모듈 생성:", MODULE)

if str(BASE) not in sys.path:
    sys.path.append(str(BASE))
from hm_data import prepare_data

root, yaml_path = prepare_data(ZIP)

weights = PROJECT / NAME / "weights"


def load_epoch(pt):
    try:
        return torch.load(pt, map_location="cpu", weights_only=False).get("epoch", -1)
    except Exception as e:
        print("체크포인트 읽기 실패:", pt.name, e)
        return None


def find_checkpoint():
    if not weights.exists():
        return None, None
    periodic = sorted(
        weights.glob("epoch*.pt"),
        key=lambda p: int(re.findall(r"\d+", p.stem)[0]),
        reverse=True,
    )
    for pt in [weights / "last.pt", *periodic]:
        if pt.exists():
            ep = load_epoch(pt)
            if ep is not None:
                return pt, ep
    return None, None


ckpt, ep = find_checkpoint()

if ckpt is not None and ep == -1:
    print("이미 학습이 끝난 run입니다:", PROJECT / NAME)
elif ckpt is not None:
    print(f"이어서 학습: {ckpt.name} (완료된 epoch {ep + 1}/{EPOCHS})")
    YOLO(str(ckpt)).train(resume=True)
else:
    print("처음부터 학습")
    YOLO(MODEL).train(
        data=str(yaml_path),
        epochs=EPOCHS,
        patience=30,
        imgsz=640,
        batch=32,
        workers=2,
        device=0,
        project=str(PROJECT),
        name=NAME,
        exist_ok=True,
        save_period=2,
        seed=42,
    )

best = YOLO(str(weights / "best.pt"))
m = best.val(data=str(yaml_path), split="test", imgsz=640)
for j, c in enumerate(m.box.ap_class_index):
    print(f"{best.names[int(c)]:<12} mAP50={m.box.ap50[j]:.3f}  mAP50-95={m.box.ap[j]:.3f}")

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
이미 풀려 있음: /content/data
root: /content/data/HM_img2
train  이미지 20885  라벨 20885
valid  이미지 1531  라벨 1531
test   이미지 356  라벨 356
이어서 학습: last.pt (완료된 epoch 35/100)
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/HM_img2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript,

In [8]:
import json, shutil, subprocess, sys, time
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4", "onnx", "onnxslim", "onnxruntime"], check=True)

import cv2
import numpy as np
import onnxruntime as ort
import torch
from google.colab import drive, files
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/HM_img2")
ZIP = BASE / "HM_img2_640.zip"
NAME = "HM_yolo26n"
RUN = BASE / "runs" / NAME
IMGSZ = 640
OUT = Path(f"/content/export/{NAME}")

if str(BASE) not in sys.path:
    sys.path.append(str(BASE))
from hm_data import prepare_data

root, yaml_path = prepare_data(ZIP)

shutil.rmtree(OUT, ignore_errors=True)
for d in ["weights", "onnx", "deepstream", "results"]:
    (OUT / d).mkdir(parents=True)

for f in ["best.pt", "last.pt"]:
    if (RUN / "weights" / f).exists():
        shutil.copy(RUN / "weights" / f, OUT / "weights" / f)
for f in RUN.iterdir():
    if f.is_file() and f.suffix in {".png", ".jpg", ".csv", ".yaml"}:
        shutil.copy(f, OUT / "results" / f.name)
shutil.copy(yaml_path, OUT / "data.yaml")

pt = OUT / "weights" / "best.pt"
model = YOLO(str(pt))
names = [model.names[i] for i in range(len(model.names))]
device = 0 if torch.cuda.is_available() else "cpu"
metrics = {"model": NAME, "imgsz": IMGSZ, "classes": names, "val": {}, "test": {}, "conf_threshold": {}}


def collect(m, key, with_threshold=False):
    for j, c in enumerate(m.box.ap_class_index):
        n = names[int(c)]
        metrics[key][n] = {
            "P": round(float(m.box.p[j]), 3),
            "R": round(float(m.box.r[j]), 3),
            "mAP50": round(float(m.box.ap50[j]), 3),
            "mAP50-95": round(float(m.box.ap[j]), 3),
        }
        if with_threshold:
            k = int(np.argmax(m.box.f1_curve[j]))
            metrics["conf_threshold"][n] = round(float(m.box.px[k]), 2)


common = dict(data=str(yaml_path), imgsz=IMGSZ, device=device, plots=False, verbose=False, project="/content/export_val", exist_ok=True)
collect(model.val(split="val", name="val", **common), "val", with_threshold=True)
collect(model.val(split="test", name="test", **common), "test")

onnx_tmp = Path(YOLO(str(pt)).export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, device="cpu"))
onnx_path = OUT / "onnx" / "best.onnx"
shutil.move(str(onnx_tmp), onnx_path)

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0].name
imgs = {p.stem: p for p in (root / "valid" / "images").iterdir()}
sample = next(
    str(imgs[l.stem])
    for l in (root / "valid" / "labels").glob("*.txt")
    if l.stem in imgs and l.read_text().startswith("0 ")
)
x = LetterBox((IMGSZ, IMGSZ), auto=False)(image=cv2.imread(sample))
x = np.ascontiguousarray(x[..., ::-1].transpose(2, 0, 1)[None]).astype(np.float32) / 255
y = sess.run(None, {inp: x})[0]
metrics["onnx"] = {"input": inp, "output_shape": list(y.shape)}
if y.ndim == 3 and y.shape[-1] == 6:
    metrics["onnx"]["dets_conf>0.25_onnx"] = int((y[0, :, 4] > 0.25).sum())
    metrics["onnx"]["dets_conf>0.25_pt"] = len(YOLO(str(pt)).predict(sample, conf=0.25, imgsz=IMGSZ, verbose=False)[0].boxes)
for _ in range(3):
    sess.run(None, {inp: x})
t = time.perf_counter()
for _ in range(20):
    sess.run(None, {inp: x})
metrics["onnx"]["colab_cpu_ms"] = round((time.perf_counter() - t) / 20 * 1000, 1)

ds_repo = Path("/content/DeepStream-Yolo")
if not ds_repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/marcoslucianops/DeepStream-Yolo", str(ds_repo)], check=True)
work = Path("/content/ds_export")
shutil.rmtree(work, ignore_errors=True)
work.mkdir()
shutil.copy(pt, work / "best.pt")
shutil.copy(ds_repo / "utils" / "export_yolo26.py", work)
try:
    subprocess.run([sys.executable, "export_yolo26.py", "-w", "best.pt", "-s", str(IMGSZ), "--simplify"], cwd=work, check=True)
    shutil.copy(next(work.glob("*.onnx")), OUT / "deepstream" / "best_ds.onnx")
    metrics["deepstream_onnx"] = "ok"
except Exception as e:
    metrics["deepstream_onnx"] = f"failed: {e}"
    print("DeepStream용 ONNX 생성 실패:", e)

(OUT / "deepstream" / "labels.txt").write_text("\n".join(names) + "\n")
th = metrics["conf_threshold"]
class_attrs = "\n".join(f"[class-attrs-{i}]\npre-cluster-threshold={th.get(n, 0.25)}\n" for i, n in enumerate(names))
(OUT / "deepstream" / "config_infer_primary_yolo26.txt").write_text(f"""[property]
gpu-id=0
net-scale-factor=0.0039215697906911373
model-color-format=0
onnx-file=best_ds.onnx
model-engine-file=best_ds.onnx_b1_gpu0_fp16.engine
labelfile-path=labels.txt
batch-size=1
network-mode=2
num-detected-classes={len(names)}
interval=0
gie-unique-id=1
process-mode=1
network-type=0
cluster-mode=4
maintain-aspect-ratio=1
symmetric-padding=1
parse-bbox-func-name=NvDsInferParseYolo
custom-lib-path=libnvdsinfer_custom_impl_Yolo.so
engine-create-func-name=NvDsInferYoloCudaEngineGet

[class-attrs-all]
pre-cluster-threshold={min(th.values()) if th else 0.25}
topk=300

{class_attrs}""")

run_sh = OUT / "deepstream" / "run_deepstream.sh"
run_sh.write_text("""#!/bin/bash
set -e
cd "$(dirname "$0")"
if [ ! -f libnvdsinfer_custom_impl_Yolo.so ]; then
  [ -d DeepStream-Yolo ] || git clone --depth 1 https://github.com/marcoslucianops/DeepStream-Yolo
  export CUDA_VER=${CUDA_VER:-12.6}
  make -C DeepStream-Yolo/nvdsinfer_custom_impl_Yolo clean
  make -C DeepStream-Yolo/nvdsinfer_custom_impl_Yolo
  cp DeepStream-Yolo/nvdsinfer_custom_impl_Yolo/libnvdsinfer_custom_impl_Yolo.so .
fi
if [ "${1:-usb}" = "csi" ]; then
  SRC="nvarguscamerasrc ! video/x-raw(memory:NVMM),width=1280,height=720,framerate=30/1"
else
  SRC="v4l2src device=/dev/video0 ! video/x-raw,width=1280,height=720 ! videoconvert ! nvvideoconvert ! video/x-raw(memory:NVMM),format=NV12"
fi
gst-launch-1.0 $SRC ! m.sink_0 nvstreammux name=m batch-size=1 width=1280 height=720 ! nvinfer config-file-path=config_infer_primary_yolo26.txt ! nvvideoconvert ! nvdsosd ! nv3dsink
""")
run_sh.chmod(0o755)

(OUT / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
(OUT / "README.txt").write_text(f"""{NAME} export

weights/best.pt, last.pt       학습 가중치
onnx/best.onnx                 onnxruntime용 (UNO Q 웹서버 등). 입력 {inp} 1x3x{IMGSZ}x{IMGSZ} RGB 0~1, letterbox
                               출력 {list(y.shape)}. 마지막 차원이 6이면 [x1, y1, x2, y2, conf, class], 입력 픽셀 좌표, NMS 불필요
deepstream/                    Jetson DeepStream용. ./run_deepstream.sh [usb|csi]
                               처음 실행 시 libnvdsinfer_custom_impl_Yolo.so를 보드에서 빌드 (CUDA_VER을 JetPack에 맞게 지정)
results/                       학습 곡선, 혼동 행렬, results.csv, args.yaml
metrics.json                   val/test 클래스별 성능, 클래스별 권장 conf, ONNX 확인 결과
data.yaml                      클래스 정의
""")

dest = BASE / "export" / NAME
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(OUT, dest)
zip_path = shutil.make_archive(f"/content/{NAME}_export", "zip", OUT)
shutil.copy(zip_path, BASE / "export" / f"{NAME}_export.zip")

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("드라이브 저장:", dest)
files.download(zip_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
이미 풀려 있음: /content/data
root: /content/data/HM_img2
train  이미지 20885  라벨 20885
valid  이미지 1531  라벨 1531
test   이미지 356  라벨 356
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,376,006 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1828.7±480.0 MB/s, size: 71.0 KB)
val: Scanning /content/data/HM_img2/valid/labels.cache... 1531 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1531/1531 169.0Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 1341, len(boxes) = 1745. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>